# Stage 1b — Geometric Duplicate Verification

**Why this replaces Stage 1's grouping.**

Stage 1 grouped images by thresholding ResNet50 ImageNet cosine similarity. Analysis of
the resulting embeddings shows that threshold cannot do the job:

| quantity | value |
|---|---|
| within-class pooled cosine, median | 0.560 |
| within-class pooled cosine, p99 | 0.934 |
| different-class baseline, median | 0.759 |

The p99 sits at 0.934, meaning ~5,800 pairs *per class* exceed 0.93 whether or not they
share a source photo. Any threshold in that regime performs **percolation**, not duplicate
detection: at tau=0.87 the average node degree is ~27 and connected components collapse
into a giant blob (214 groups, one of 238 images spanning 4 classes).

The verification gate also gave a false PASS. `test_above_tau_frac == 0` is *tautological*
under single-linkage clustering — no cross-group edge can exceed tau by construction.
Only the baseline-margin criterion carried information.

**The fix.** Semantic similarity answers "do these look like the same condition?" We need
"are these the same photograph?" — a geometric question. This notebook uses embeddings only
as a cheap *candidate generator*, then verifies each candidate with **ORB keypoint matching
plus RANSAC homography estimation**. Two augmented variants of one photo share a projective
transform and yield many geometric inliers; two different photos of the same disease yield
almost none, however similar they look.

The inlier threshold is **calibrated empirically** against synthesised known-variant pairs
rather than guessed — that calibration is itself a defensible methods figure.

**Requires:** `emb.npy`, `emb_mirror.npy`, `manifest.csv` from Stage 1 (no re-embedding).

In [1]:
import os, json, math, random, itertools, warnings
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import cv2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

warnings.filterwarnings("ignore")

STAGE1   = Path("/kaggle/input/notebooks/shafinbubt/stage-1/stage1/")   # or /kaggle/working/stage1
OUT_DIR  = Path("/kaggle/working/stage1b")
OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / "figs").mkdir(exist_ok=True)

SEED          = 42
ORB_SIZE      = 512     # longest side for ORB; bigger = more keypoints = slower
ORB_FEATURES  = 1500
BLOCK_TAU     = 0.85    # candidate generator, deliberately LOOSE (recall-oriented)
BLOCK_TAU_X   = 0.90    # stricter for cross-class candidates
LOWE_RATIO    = 0.75
RANSAC_THRESH = 4.0

random.seed(SEED); np.random.seed(SEED)

m = pd.read_csv(STAGE1 / "manifest.csv")
E  = np.load(STAGE1 / "emb.npy")
EM = np.load(STAGE1 / "emb_mirror.npy")
CLASSES = sorted(m["label"].unique())
print(f"{len(m)} images | {len(CLASSES)} classes | embeddings {E.shape}")

10800 images | 10 classes | embeddings (10800, 2048)


## 1. ORB descriptor cache

Computed once per image and reused across every candidate pair. This is what makes
pairwise geometric verification affordable.

In [2]:
orb = cv2.ORB_create(nfeatures=ORB_FEATURES, scaleFactor=1.2, nlevels=8)

def load_gray(path, size=ORB_SIZE):
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None
    h, w = img.shape
    s = size / max(h, w)
    if s < 1.0:
        img = cv2.resize(img, (int(w * s), int(h * s)), interpolation=cv2.INTER_AREA)
    return cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(img)

def describe_arr(g):
    if g is None:
        return None, None
    kp, des = orb.detectAndCompute(g, None)
    if des is None or len(kp) < 8:
        return None, None
    return np.float32([k.pt for k in kp]), des

# ORB's rBRIEF descriptors are rotation-invariant but NOT mirror-invariant. A horizontally
# flipped variant of the same photo scores in the same inlier range as an unrelated image,
# so every image is described twice -- as-is and mirrored -- and pair scores take the max
# over both pairings. Without this, every flipped duplicate escapes detection.
KP, DES   = [None] * len(m), [None] * len(m)
KPF, DESF = [None] * len(m), [None] * len(m)
for i, _p in enumerate(m["path"]):
    g = load_gray(_p)
    KP[i],  DES[i]  = describe_arr(g)
    KPF[i], DESF[i] = describe_arr(None if g is None else cv2.flip(g, 1))
    if (i + 1) % 1000 == 0:
        print(f"  described {i+1}/{len(m)}")
n_ok = sum(d is not None for d in DES)
print(f"ORB descriptors: {n_ok}/{len(m)} images ({n_ok/len(m):.1%})")

  described 1000/10800
  described 2000/10800
  described 3000/10800
  described 4000/10800
  described 5000/10800
  described 6000/10800
  described 7000/10800
  described 8000/10800
  described 9000/10800
  described 10000/10800
ORB descriptors: 10800/10800 images (100.0%)


## 2. Geometric verification primitive

Lowe ratio test for putative matches, then RANSAC homography. The **inlier count** is the
same-source evidence. A homography is the right model: rotation, scale, flip, shear,
translation and crop — the whole standard augmentation repertoire — are all projective.

In [3]:
bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=False)

def inliers_arr(kp1, des1, kp2, des2):
    """RANSAC homography inlier count for one descriptor pairing."""
    if des1 is None or des2 is None:
        return 0
    try:
        knn = bf.knnMatch(des1, des2, k=2)
    except cv2.error:
        return 0
    good = [a for a, b in (p for p in knn if len(p) == 2)
            if a.distance < LOWE_RATIO * b.distance]
    if len(good) < 8:
        return 0
    src = np.float32([kp1[g.queryIdx] for g in good]).reshape(-1, 1, 2)
    dst = np.float32([kp2[g.trainIdx] for g in good]).reshape(-1, 1, 2)
    H, mask = cv2.findHomography(src, dst, cv2.RANSAC, RANSAC_THRESH, maxIters=2000)
    return 0 if mask is None else int(mask.sum())

def geo_inliers(i, j):
    """Mirror-aware inlier count between images i and j."""
    return max(inliers_arr(KP[i], DES[i], KP[j],  DES[j]),
               inliers_arr(KP[i], DES[i], KPF[j], DESF[j]))

## 3. Calibrate the inlier threshold

We synthesise the augmentations the vendor plausibly used, applied to real images from this
dataset, so we know ground truth. Two distributions get measured:

- **positives** — an image vs its own synthetic variant (same source, by construction)
- **negatives** — two distinct images of the *same class* that Stage 1's embedding
  similarity rated highly (the hardest possible negatives)

The threshold goes in the gap. If there is no gap, geometric verification does not work on
this data either, and that must be known before anything else is built.

In [4]:
def synth_variant(img, rng):
    h, w = img.shape[:2]
    ang   = rng.uniform(-30, 30)
    scale = rng.uniform(0.85, 1.20)
    M = cv2.getRotationMatrix2D((w / 2, h / 2), ang, scale)
    M[0, 2] += rng.uniform(-0.08, 0.08) * w
    M[1, 2] += rng.uniform(-0.08, 0.08) * h
    out = cv2.warpAffine(img, M, (w, h), borderMode=cv2.BORDER_REFLECT)
    if rng.random() < 0.5:
        out = cv2.flip(out, 1)
    out = np.clip(out.astype(np.float32) * rng.uniform(0.75, 1.30)
                  + rng.uniform(-25, 25), 0, 255).astype(np.uint8)
    if rng.random() < 0.3:
        out = cv2.GaussianBlur(out, (3, 3), 0)
    return out

rng = np.random.default_rng(SEED)
cal_ids = rng.choice(np.where([d is not None for d in DES])[0], size=200, replace=False)

pos = []
for i in cal_ids:
    g = load_gray(m.at[i, "path"])
    for _ in range(2):
        v = synth_variant(g, rng)
        k2, d2 = describe_arr(v)
        k2f, d2f = describe_arr(cv2.flip(v, 1))
        pos.append(max(inliers_arr(KP[i], DES[i], k2, d2),
                       inliers_arr(KP[i], DES[i], k2f, d2f)))

def ps(A, AM, B, BM):
    s = A @ B.T
    np.maximum(s, A @ BM.T, out=s); np.maximum(s, AM @ B.T, out=s)
    np.maximum(s, AM @ BM.T, out=s); return s

neg = []
for i in cal_ids[:150]:
    same = m.index[(m["label"] == m.at[i, "label"]) & (m.index != i)].to_numpy()
    S = ps(E[[i]], EM[[i]], E[same], EM[same])[0]
    for j in same[np.argsort(-S)[:3]]:          # hardest negatives available
        neg.append(geo_inliers(int(i), int(j)))

pos, neg = np.array(pos), np.array(neg)
print(f"positives  n={len(pos)}  median={np.median(pos):.0f}  p5={np.percentile(pos,5):.0f}")
print(f"negatives  n={len(neg)}  median={np.median(neg):.0f}  p95={np.percentile(neg,95):.0f}  max={neg.max()}")

cands = np.arange(5, 81)
f1s = []
for t in cands:
    tp = (pos >= t).sum(); fp = (neg >= t).sum(); fn = (pos < t).sum()
    f1s.append(2 * tp / max(2 * tp + fp + fn, 1))
INLIER_TAU = int(cands[int(np.argmax(f1s))])
tpr = (pos >= INLIER_TAU).mean(); fpr = (neg >= INLIER_TAU).mean()
print(f"\nCalibrated INLIER_TAU = {INLIER_TAU}   TPR={tpr:.3f}  FPR={fpr:.3f}  F1={max(f1s):.3f}")
SEPARABLE = tpr > 0.85 and fpr < 0.05
print("SEPARABLE" if SEPARABLE else
      ">> WARNING: distributions overlap. Do not trust the grouping; see notes at end.")

positives  n=400  median=642  p5=427
negatives  n=450  median=442  p95=720  max=1011

Calibrated INLIER_TAU = 80   TPR=1.000  FPR=0.922  F1=0.658
>> WARNING: distributions overlap. Do not trust the grouping; see notes at end.


In [5]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
bins = np.linspace(0, max(pos.max(), neg.max(), 1), 45)
ax[0].hist(pos, bins=bins, alpha=.75, label="same source (synthetic variant)", color="#4C72B0")
ax[0].hist(neg, bins=bins, alpha=.65, label="different source, same class", color="#C44E52")
ax[0].axvline(INLIER_TAU, color="k", ls="--", label=f"threshold {INLIER_TAU}")
ax[0].set_xlabel("RANSAC inliers"); ax[0].set_ylabel("count")
ax[0].set_title("Calibration of geometric duplicate criterion"); ax[0].legend()
ax[1].plot(cands, f1s, color="#4C72B0"); ax[1].axvline(INLIER_TAU, color="k", ls="--")
ax[1].set_xlabel("inlier threshold"); ax[1].set_ylabel("F1"); ax[1].set_title("Threshold selection")
plt.tight_layout(); plt.savefig(OUT_DIR / "figs" / "fig2_calibration.png", dpi=200); plt.show()

## 4. Candidate generation, then verification

Embeddings are used only for **recall** here. A loose 0.85 block keeps the pair count
tractable while retaining essentially every true duplicate — true variants sit far above
0.85 in this space. Precision comes entirely from the geometric stage.

In [6]:
pairs = []
for c in CLASSES:
    ids = m.index[m["label"] == c].to_numpy()
    S = ps(E[ids], EM[ids], E[ids], EM[ids]); np.fill_diagonal(S, -1)
    ii, jj = np.where(S >= BLOCK_TAU)
    pairs += [(int(ids[a]), int(ids[b])) for a, b in zip(ii, jj) if a < b]
n_within = len(pairs)

for s in range(0, len(m), 512):
    sl = np.arange(s, min(s + 512, len(m)))
    S = ps(E[sl], EM[sl], E, EM)
    for r, i in enumerate(sl):
        S[r, i] = -1
        for j in np.where(S[r] >= BLOCK_TAU_X)[0]:
            if j > i and m.at[i, "y"] != m.at[int(j), "y"]:
                pairs.append((int(i), int(j)))
print(f"candidate pairs: {n_within} within-class + {len(pairs)-n_within} cross-class "
      f"= {len(pairs)} to verify")

verified = []
for n, (i, j) in enumerate(pairs):
    v = geo_inliers(i, j)
    if v >= INLIER_TAU:
        verified.append((i, j, v))
    if (n + 1) % 5000 == 0:
        print(f"  verified {n+1}/{len(pairs)} -> {len(verified)} duplicates so far")

print(f"\ngeometrically verified duplicate pairs: {len(verified)} "
      f"({len(verified)/max(len(pairs),1):.1%} of candidates)")
vdf = pd.DataFrame(verified, columns=["i", "j", "inliers"])
vdf["label_i"] = m.loc[vdf["i"], "label"].values
vdf["label_j"] = m.loc[vdf["j"], "label"].values
vdf.to_csv(OUT_DIR / "verified_duplicate_pairs.csv", index=False)
x_cls = vdf[vdf["label_i"] != vdf["label_j"]]
print(f"of which cross-class (same photo, two labels = vendor label noise): {len(x_cls)}")
if len(x_cls):
    print(x_cls.groupby(["label_i", "label_j"]).size().sort_values(ascending=False).head(8))

candidate pairs: 232452 within-class + 423 cross-class = 232875 to verify
  verified 5000/232875 -> 4678 duplicates so far
  verified 10000/232875 -> 9317 duplicates so far
  verified 15000/232875 -> 13957 duplicates so far
  verified 20000/232875 -> 18730 duplicates so far
  verified 25000/232875 -> 23288 duplicates so far
  verified 30000/232875 -> 27671 duplicates so far
  verified 35000/232875 -> 32077 duplicates so far
  verified 40000/232875 -> 36510 duplicates so far
  verified 45000/232875 -> 40832 duplicates so far
  verified 50000/232875 -> 45225 duplicates so far
  verified 55000/232875 -> 49536 duplicates so far
  verified 60000/232875 -> 53684 duplicates so far
  verified 65000/232875 -> 57647 duplicates so far
  verified 70000/232875 -> 61408 duplicates so far
  verified 75000/232875 -> 65812 duplicates so far
  verified 80000/232875 -> 70195 duplicates so far
  verified 85000/232875 -> 74341 duplicates so far
  verified 90000/232875 -> 78403 duplicates so far
  verified 

## 5. Group by transitive closure over *verified* pairs

Chaining is no longer a hazard: every edge is a proven same-source relation, and being the
same photograph is genuinely transitive. Cross-class verified pairs are merged too — one
photograph must not straddle the split regardless of how it was labelled.

In [7]:
A = coo_matrix((np.ones(len(vdf)), (vdf["i"], vdf["j"])), shape=(len(m), len(m)))
n_groups, gid = connected_components(A, directed=False)
m["group_id"] = gid
sz = m.groupby("group_id").size()
spans = m.groupby("group_id")["label"].nunique()

print(f"source groups          : {n_groups}")
print(f"mean variants / group  : {sz.mean():.2f}")
print(f"median / max group size: {int(sz.median())} / {int(sz.max())}")
print(f"singletons             : {int((sz==1).sum())}")
print(f"groups spanning classes: {int((spans>1).sum())}  (should be small)")
print(f"\nunique sources per class:")
print(m.groupby("label")["group_id"].nunique().to_string())

source groups          : 248
mean variants / group  : 43.55
median / max group size: 38 / 206
singletons             : 14
groups spanning classes: 1  (should be small)

unique sources per class:
label
Alopecia Areata          28
Contact Dermatitis        8
Folliculitis             28
Head Lice                13
Lichen Planus            18
Male Pattern Baldness    31
Psoriasis                29
Seborrheic Dermatitis    23
Telogen Effluvium        32
Tinea Capitis            39


## 6. Honest verification gate

Stage 1's gate was tautological. This one is not. For each held-out medoid we take its top
embedding neighbours in train and **geometrically verify them**. The gate asks the real
question: *is any evaluation image the same photograph as any training image?*

In [8]:
from sklearn.model_selection import train_test_split

med = {}
for g, gr in m.groupby("group_id"):
    ids = gr.index.to_numpy()
    if len(ids) == 1:
        med[g] = int(ids[0]); continue
    S = ps(E[ids], EM[ids], E[ids], EM[ids]); np.fill_diagonal(S, 0)
    med[g] = int(ids[S.mean(1).argmax()])
m["is_medoid"] = m.index.isin(set(med.values()))

gtab = m.groupby("group_id").agg(y=("y", "first")).reset_index()
vc = gtab["y"].value_counts()
gtab = gtab[gtab["y"].isin(vc[vc >= 5].index)]        # need >=5 groups to stratify
tr_g, tmp = train_test_split(gtab, train_size=0.70, stratify=gtab["y"], random_state=SEED)
val_g, te_g = train_test_split(tmp, train_size=0.5, stratify=tmp["y"], random_state=SEED)
split_of = {**{g: "train" for g in tr_g["group_id"]},
            **{g: "val" for g in val_g["group_id"]},
            **{g: "test" for g in te_g["group_id"]}}
m["split"] = m["group_id"].map(split_of)
m["role"] = np.where(m["split"] == "train", "train_use",
             np.where(m["is_medoid"] & m["split"].notna(), m["split"] + "_use", "discard"))

train_ids = m.index[m["role"] == "train_use"].to_numpy()
eval_ids  = m.index[m["role"].isin(["test_use", "val_use"])].to_numpy()

TOPK, residual = 20, 0
for i in eval_ids:
    S = ps(E[[i]], EM[[i]], E[train_ids], EM[train_ids])[0]
    for j in train_ids[np.argsort(-S)[:TOPK]]:
        if geo_inliers(int(i), int(j)) >= INLIER_TAU:
            residual += 1; break

leak_rate = residual / max(len(eval_ids), 1)
print(f"eval images with a VERIFIED duplicate in train: {residual}/{len(eval_ids)} "
      f"= {leak_rate:.2%}")
n_test = int((m['role'] == 'test_use').sum())
PASS = leak_rate < 0.005 and n_test >= 150
print("\n" + "=" * 66)
print(f"train {int((m['role']=='train_use').sum())} | "
      f"val {int((m['role']=='val_use').sum())} | test {n_test}")
print("GATE: " + ("PASS — proceed to Stage 2" if PASS else
      "REVIEW — see decision notes below"))
print("=" * 66)

m[["path", "label", "y", "vendor_split", "group_id", "is_medoid", "split", "role"]] \
    .to_csv(OUT_DIR / "manifest_v2.csv", index=False)
json.dump({"inlier_tau": INLIER_TAU, "tpr": float(tpr), "fpr": float(fpr),
           "n_groups": int(n_groups), "mean_group_size": float(sz.mean()),
           "verified_pairs": int(len(vdf)), "cross_class_verified": int(len(x_cls)),
           "residual_leak_rate": float(leak_rate),
           "n_train": int(len(train_ids)), "n_test": n_test},
          open(OUT_DIR / "report_v2.json", "w"), indent=2)

eval images with a VERIFIED duplicate in train: 1/75 = 1.33%

train 7690 | val 37 | test 38
GATE: REVIEW — see decision notes below


## Decision notes — read before Stage 2

**If `n_groups` is ~1000-1300 and the gate passes.** The 1200x9 hypothesis holds. Use
**grouped stratified 5-fold cross-validation**, not the single 70/15/15 split above: with
~1000 unique sources a single test fold is ~150 images and one lucky split will not
convince a board. Five folds give you mean +/- std, which is a much stronger result.

**If `n_groups` is under ~400.** The dataset contains far fewer independent photographs
than its 10,800 files suggest, and no split can manufacture statistical power that is not
there. That is a publishable finding, but it will not carry an implementation thesis on its
own. Switch datasets — HAM10000, ISIC, or Fitzpatrick17k are large, properly curated
dermatology sets with no leakage problem, and the XAI half of your methodology, your
architecture diagrams and most of your literature review transfer unchanged.

**If calibration was not `SEPARABLE`.** Geometric verification failed, which usually means
the images are too low-resolution or too texture-poor for ORB. Raise `ORB_SIZE` to 768 and
retry once; if it still overlaps, treat the dataset as unusable and switch.

**Watermarks.** The Stage 1 border-only probe scored 0.364 against 0.100 chance — a 3.6x
lift — but on only 33 test images, so the confidence interval is wide (~0.20-0.53). Re-run
that probe on this larger test set before concluding. If the lift holds, Stage 2 must
centre-crop, and the before/after comparison becomes a genuine contribution.